# 003 — PSHA setup & analyses

Sets up and runs the six WP1 PSHA analyses on the ESHM20 model: **AvgSA 0–3** and
**AvgSA 0–6**, each at truncation level (epsilon) **3, 4 and 5**.

1. Verifies the manually placed hazard-model prerequisites in `hazard_models/eshm20/wp1/`.
2. Writes the site model into `wp1/` from `results/01_site_selection/` (the source of truth).
3. Generates the six job configs `config_AvgSA_0{3,6}_psha_eps{3,4,5}.ini`.
4. Runs each calculation with `oq engine --run`, **skipping any whose inputs are unchanged**.
5. Records each `calc_id` in `wp1/psha_manifest.json` so the datastores can be obtained
   programmatically downstream instead of by hardcoded integer.

## Manually placed prerequisites

The hazard model files in `hazard_models/eshm20/wp1/` are **not produced by this notebook**.
They were placed there by hand (copied from `hazard_models/eshm20/regional_avgSA/`):

| File | Provenance |
| --- | --- |
| `source_model_logic_tree_eshm20.xml` | **untouched** from ESHM20 |
| `source_models/` (6 subfolders, ~34 MB) | **untouched** from ESHM20 |
| `gmpe_logic_tree_AvgSA_0to3_median_branch.xml` | **manually simplified** — single median branch per TRT |
| `gmpe_logic_tree_AvgSA_0to6_median_branch.xml` | **manually simplified** — single median branch per TRT |

The two GMM logic trees differ only in `avg_periods` (10 evenly spaced periods over
0–3 s and 0–6 s respectively). Both use `GenericGmpeAvgSA` with the `clemett_*`
correlation functions, so the installed OpenQuake engine must implement those and
ship their coefficients.

## Dependencies

**Upstream:** `001-site_selection.ipynb` → `results/01_site_selection/site_model_all_sites.csv`

**Downstream:** `004-psha_results.ipynb`, `020-disaggregation.ipynb` — both consume the
`calc_id`s recorded here via `oq_runner.load_calc_ids(...)`.

## Runtime

These are **long** calculations (a 21.8 MB point-source model × 60 sites, six times over).
`DRY_RUN = True` is the default so the notebook can be run end-to-end to inspect the
configs without launching anything.

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [2]:
import pandas as pd

from phd_project.config import config
from phd_project.scripts.WP1_ground_motion_set import oq_runner

cfg = config.load_config()

In [ ]:
# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------
WP1_DIR = cfg["hazard_models"]["eshm20_wp1"]
MANIFEST_FP = cfg["hazard_models"]["eshm20_wp1_psha_manifest"]
SITE_MODEL_SRC = cfg["results"]["site_selection"] / "site_model_all_sites.csv"

# IM period range -> the (manually simplified) GMM logic tree defining it.
IM_DEFINITIONS = oq_runner.GMPE_LOGIC_TREES  # {"03": ..., "06": ...}

# Truncation levels (epsilon) to run each IM definition at. One PSHA is set up and
# run per (IM definition x truncation level) pair.
TRUNCATION_LEVELS = [3, 4, 5]

EXPECTED_N_SITES = 60

# DRY_RUN:    write configs and report what would run, but launch nothing.
# FORCE_RERUN: re-run every analysis even when its inputs are unchanged.
# NEW_WINDOW:  give each calculation its own console window so its progress can be
#              watched (Windows only; output is teed to WP1_DIR/logs/<name>.log
#              either way). Runs stay sequential regardless.
DRY_RUN = False
FORCE_RERUN = False
NEW_WINDOW = True
# -----------------------------------------------------------------------------

print(f"wp1 dir:   {WP1_DIR}")
print(f"manifest:  {MANIFEST_FP}")
print(f"analyses:  {len(IM_DEFINITIONS) * len(TRUNCATION_LEVELS)} "
      f"(AvgSA {sorted(IM_DEFINITIONS)} x eps {TRUNCATION_LEVELS})")
print(f"DRY_RUN={DRY_RUN}, FORCE_RERUN={FORCE_RERUN}, NEW_WINDOW={NEW_WINDOW}")

wp1 dir:   C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1
manifest:  C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\psha_manifest.json
analyses:  6 (AvgSA ['03', '06'] x eps [3, 4, 5])
DRY_RUN=True, FORCE_RERUN=False, NEW_WINDOW=True


## 1. Verify prerequisites & write the site model

Checks the manually placed files are all present, then writes the site model into
`wp1/` from `results/01_site_selection/`.

In [4]:
required = [
    WP1_DIR / oq_runner.SOURCE_MODEL_LOGIC_TREE_FILE,
    *(WP1_DIR / lt for lt in IM_DEFINITIONS.values()),
    *(WP1_DIR / "source_models" / d for d in
      ["asm_v12e", "deep_v12e", "fsm_v09", "interface_v12b", "ssm_v09", "volcanic_v12e"]),
]

# Report every missing path at once rather than failing on the first.
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing manually placed prerequisites in wp1/ (see the header cell):\n"
        + "\n".join(f"  - {p}" for p in missing)
    )
print(f"All {len(required)} prerequisites present in {WP1_DIR.name}/")

All 9 prerequisites present in wp1/


In [5]:
site_model = pd.read_csv(SITE_MODEL_SRC)
assert len(site_model) == EXPECTED_N_SITES, (
    f"Expected {EXPECTED_N_SITES} sites in {SITE_MODEL_SRC.name}, found {len(site_model)}"
)
print(f"Loaded {len(site_model)} sites from {SITE_MODEL_SRC}")
print(f"Columns: {list(site_model.columns)}")

site_model_fp = cfg["hazard_models"]["eshm20_wp1_site_model"]
text = site_model.to_csv(index=False, lineterminator="\n")
# Skip a no-op rewrite: a churning mtime would not change the fingerprint (the
# site model is hashed by content) but it keeps the wp1/ copy honest about when
# it last actually diverged from /results.
if site_model_fp.is_file() and site_model_fp.read_text() == text:
    print(f"{site_model_fp.name} already up to date.")
else:
    site_model_fp.write_text(text)
    print(f"Wrote {site_model_fp}")

site_model.head()

Loaded 60 sites from C:\Users\clemettn\Documents\phd\results\01_site_selection\site_model_all_sites.csv
Columns: ['lat', 'lon', 'region', 'vs30', 'vs30measured', 'xvf', 'z1pt0', 'z2pt5']
site_model_all_sites.csv already up to date.


,lat,lon,region,vs30,vs30measured,xvf,z1pt0,z2pt5
0,36.1,-5.31787,0,800,True,150.0,31.07,0.57
1,38.4,-0.51787,0,800,True,150.0,31.07,0.57
2,48.5,9.08213,0,800,True,150.0,31.07,0.57
3,47.2,18.48213,0,800,True,150.0,31.07,0.57
4,37.5,40.88213,0,800,True,150.0,31.07,0.57


## 2. Write the six PSHA configs

All six are rendered from one template in `oq_runner`, so they differ only in
`description`, `gsim_logic_tree_file` and `truncation_level`.

In [6]:
analyses = {}
for im, gmpe_lt in IM_DEFINITIONS.items():
    for eps in TRUNCATION_LEVELS:
        name = oq_runner.analysis_name(im, eps)
        fp = oq_runner.write_config(
            WP1_DIR / oq_runner.config_name(im, eps),
            description=oq_runner.analysis_description(im, eps),
            gsim_logic_tree_file=gmpe_lt,
            truncation_level=eps,
            im_upper=oq_runner.im_upper(im),
        )
        analyses[name] = fp
        print(f"{name:24s} -> {fp.name}")

AvgSA_03_psha_eps3       -> config_AvgSA_03_psha_eps3.ini
AvgSA_03_psha_eps4       -> config_AvgSA_03_psha_eps4.ini
AvgSA_03_psha_eps5       -> config_AvgSA_03_psha_eps5.ini
AvgSA_06_psha_eps3       -> config_AvgSA_06_psha_eps3.ini
AvgSA_06_psha_eps4       -> config_AvgSA_06_psha_eps4.ini
AvgSA_06_psha_eps5       -> config_AvgSA_06_psha_eps5.ini


## 3. Run (or reuse) the calculations

Each analysis is launched only if it has no recorded `calc_id`, its inputs have
changed since that id was recorded, or its datastore has gone missing. Runs are
sequential — the engine parallelises internally, so concurrent `oq engine` runs
would only contend for cores.

With `NEW_WINDOW = True` each calculation opens its **own console window**, so the
engine's live progress can be watched. The notebook cell blocks until that window's
calculation finishes, then moves on to the next; the window closes on completion.
Output is mirrored to `wp1/logs/<analysis>.log` either way, so progress is still
reviewable after the window has gone.

> Set `DRY_RUN = False` in the parameters cell to actually launch. Expect these
> to take a long time.

In [7]:
calc_ids = {}
for name, config_fp in analyses.items():
    calc_ids[name] = oq_runner.run_or_reuse(
        name, config_fp, WP1_DIR, MANIFEST_FP,
        force_rerun=FORCE_RERUN, dry_run=DRY_RUN, new_window=NEW_WINDOW,
    )

calc_ids

[oq] 'AvgSA_03_psha_eps3' would run config_AvgSA_03_psha_eps3.ini (dry_run).
[oq] 'AvgSA_03_psha_eps4' would run config_AvgSA_03_psha_eps4.ini (dry_run).
[oq] 'AvgSA_03_psha_eps5' would run config_AvgSA_03_psha_eps5.ini (dry_run).
[oq] 'AvgSA_06_psha_eps3' would run config_AvgSA_06_psha_eps3.ini (dry_run).
[oq] 'AvgSA_06_psha_eps4' would run config_AvgSA_06_psha_eps4.ini (dry_run).
[oq] 'AvgSA_06_psha_eps5' would run config_AvgSA_06_psha_eps5.ini (dry_run).


{'AvgSA_03_psha_eps3': None,
 'AvgSA_03_psha_eps4': None,
 'AvgSA_03_psha_eps5': None,
 'AvgSA_06_psha_eps3': None,
 'AvgSA_06_psha_eps4': None,
 'AvgSA_06_psha_eps5': None}

## 4. Manifest

`psha_manifest.json` is the record downstream notebooks read with
`oq_runner.load_calc_ids(cfg["hazard_models"]["eshm20_wp1_psha_manifest"])`.
It is small and text, so it is **git-tracked** (not DVC-tracked) — it is the
pointer that makes the DVC-tracked hazard artifacts reproducible.

In [8]:
manifest = oq_runner.load_manifest(MANIFEST_FP)
if not manifest:
    print(f"No manifest yet at {MANIFEST_FP} - run with DRY_RUN = False.")
else:
    summary = pd.DataFrame([
        {
            "analysis": name,
            "calc_id": e["calc_id"],
            "config": e["config"],
            "description": e["description"],
            "git_commit": e["_meta"]["git_commit"],
            "written_at": e["_meta"]["written_at"],
        }
        for name, e in manifest.items()
    ]).sort_values("analysis").reset_index(drop=True)
    display(summary)

No manifest yet at C:\Users\clemettn\Documents\phd\hazard_models\eshm20\wp1\psha_manifest.json - run with DRY_RUN = False.